In [7]:
import pandas as pd
import dataset
df = dataset.clean(month_end=1)
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,month
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0,1
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,1
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0,1
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0,1
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0,1


In [8]:
# Time aggregates
df['pickup_hr'] = df['tpep_pickup_datetime'].dt.hour
df['pickup_day'] = df['tpep_pickup_datetime'].dt.day
df['pickup_month'] = df['tpep_pickup_datetime'].dt.month
df['dropoff_hr'] = df['tpep_dropoff_datetime'].dt.hour
df['dropoff_day'] = df['tpep_dropoff_datetime'].dt.day
df['dropoff_month'] = df['tpep_dropoff_datetime'].dt.month

# store_and_fwd_flag
#
# This flag indicates whether the trip record was held in vehicle memory before
# sending to the vendor, aka “store and forward,” because the vehicle did not
# have a connection to the server.
# Y = store and forward trip
# N = not a store and forward trip
#
# Change flag from 'Y' or 'N' -> True or False
df['store_and_fwd_flag'] = df['store_and_fwd_flag'].map({'Y': True, 'N': False})

In [ ]:
# RatecodeID
#
# The final rate code in effect at the end of the trip.
# 1 = Standard rate
# 2 = JFK
# 3 = Newark
# 4 = Nassau or Westchester
# 5 = Negotiated fare
# 6 = Group ride
# 99 = Null/unknown
ratecode_mapping = {
    1: "Standard Rate",
    2: "JFK",
    3: "Newark",
    4: "Nassua or Westchester",
    5: "Negotiated fare",
    6: "Group Ride",
    99: "Null/unknown",
}
df['ratecode'] = df['RatecodeID'].map(ratecode_mapping)

# payment_type
#
# A numeric code signifying how the passenger paid for the trip.
# 0 = Flex Fare trip
# 1 = Credit card
# 2 = Cash
# 3 = No charge
# 4 = Dispute
# 5 = Unknown
# 6 = Voided trip
payment_type_mapping = {
    0: "Flex Fare trip",
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip",
}
df['payment_type'] = df['payment_type'].map(payment_type_mapping)

# VendorID

# A code indicating the TPEP provider that provided the record.
# 1 = Creative Mobile Technologies, LLC
# 2 = Curb Mobility, LLC
# 6 = Myle Technologies Inc
# 7 = Helix
vendor_mapping = {
    1: "Creative Mobile Technologies, LLC",
    2: "Curb Mobility, LLC",
    6: "Myle Technologies Inc",
    7: "Helix",
}
df['vendor'] = df['VendorID'].map(vendor_mapping)

In [55]:
# columns_to_drop = [
#     'RatecodeID',
#     'VendorID',
#     'Borough', # All boroughs are = 'Manhattan'
#     'PULocationID',
#     'DOLocationID',
# ]
# df = df.drop(columns=columns_to_drop)

In [ ]:
# Taxi zones
#
# .csv file needed to lookup codes
zones = pd.read_csv("../../data/taxi_zone_lookup.csv")
manhattan_ids = zones[zones["Borough"] == "Manhattan"]["LocationID"].tolist()

# Taxi zones: Pickup zone & pickup service zone
df = df[df["PULocationID"].isin(manhattan_ids)]
df = df.merge(
    zones[['LocationID', 'Zone', 'service_zone']].rename(columns={
        'Zone': 'pickup_zone',
        'service_zone': 'pickup_service_zone',
    }),
    left_on='PULocationID', 
    right_on='LocationID',
    how='left'
)
df = df.drop(columns=['LocationID'])

# Taxi zones: Dropoff zone & dropoff service zone
df = df.merge(  
    zones[['LocationID', 'Zone', 'service_zone']].rename(columns={
        'Zone': 'dropoff_zone',
        'service_zone': 'dropoff_service_zone',
    }),
    left_on='DOLocationID', 
    right_on='LocationID',
    how='left'
)
df = df.drop(columns=['LocationID'])

In [11]:
df

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,pickup_month,dropoff_hr,dropoff_day,dropoff_month,Ratecode,Vendor,PickupZone,PickupServiceZone,DropoffZone,DropoffServiceZone
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,False,186,79,Cash,...,1,1,1,1,Standard Rate,"Curb Mobility, LLC",Penn Station/Madison Sq West,Yellow Zone,East Village,Yellow Zone
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,False,140,236,Credit card,...,1,0,1,1,Standard Rate,"Creative Mobile Technologies, LLC",Lenox Hill East,Yellow Zone,Upper East Side North,Yellow Zone
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,False,236,79,Credit card,...,1,0,1,1,Standard Rate,"Creative Mobile Technologies, LLC",Upper East Side North,Yellow Zone,East Village,Yellow Zone
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,False,79,211,Credit card,...,1,0,1,1,Standard Rate,"Creative Mobile Technologies, LLC",East Village,Yellow Zone,SoHo,Yellow Zone
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,False,211,148,Credit card,...,1,0,1,1,Standard Rate,"Creative Mobile Technologies, LLC",SoHo,Yellow Zone,Lower East Side,Yellow Zone
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2573364,1,2024-01-31 23:18:48,2024-01-31 23:38:05,NaN,3.90,NaN,NaN,90,238,Flex Fare trip,...,1,23,31,1,NaN,"Creative Mobile Technologies, LLC",Flatiron,Yellow Zone,Upper West Side North,Yellow Zone
2573365,2,2024-01-31 23:45:59,2024-01-31 23:54:36,NaN,3.18,NaN,NaN,107,263,Flex Fare trip,...,1,23,31,1,NaN,"Curb Mobility, LLC",Gramercy,Yellow Zone,Yorkville West,Yellow Zone
2573366,1,2024-01-31 23:13:07,2024-01-31 23:27:52,NaN,4.00,NaN,NaN,114,236,Flex Fare trip,...,1,23,31,1,NaN,"Creative Mobile Technologies, LLC",Greenwich Village South,Yellow Zone,Upper East Side North,Yellow Zone
2573367,2,2024-01-31 23:19:00,2024-01-31 23:38:00,NaN,3.33,NaN,NaN,211,25,Flex Fare trip,...,1,23,31,1,NaN,"Curb Mobility, LLC",SoHo,Yellow Zone,Boerum Hill,Boro Zone


In [1]:
import dataset
df = dataset.clean()
df.head()

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,...,pickup_month,dropoff_hr,dropoff_day,dropoff_month,ratecode,vendor,pickup_zone,pickup_service_zone,dropoff_zone,dropoff_service_zone
0,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,False,Cash,17.7,1.0,0.5,0.00,...,1,1,1,1,Standard Rate,"Curb Mobility, LLC",Penn Station/Madison Sq West,Yellow Zone,East Village,Yellow Zone
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,False,Credit card,10.0,3.5,0.5,3.75,...,1,0,1,1,Standard Rate,"Creative Mobile Technologies, LLC",Lenox Hill East,Yellow Zone,Upper East Side North,Yellow Zone
2,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,False,Credit card,23.3,3.5,0.5,3.00,...,1,0,1,1,Standard Rate,"Creative Mobile Technologies, LLC",Upper East Side North,Yellow Zone,East Village,Yellow Zone
3,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,False,Credit card,10.0,3.5,0.5,2.00,...,1,0,1,1,Standard Rate,"Creative Mobile Technologies, LLC",East Village,Yellow Zone,SoHo,Yellow Zone
4,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,False,Credit card,7.9,3.5,0.5,3.20,...,1,0,1,1,Standard Rate,"Creative Mobile Technologies, LLC",SoHo,Yellow Zone,Lower East Side,Yellow Zone


In [7]:
df = df.astype({
    'store_and_fwd_flag': 'bool',
    'payment_type': 'category',
    'ratecode': 'category',
    'vendor': 'category',
    'pickup_zone': 'category',
    'pickup_service_zone': 'category',
    'dropoff_zone': 'category',
    'dropoff_service_zone': 'category',
})
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5198008 entries, 0 to 5198007
Data columns (total 28 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   tpep_pickup_datetime   datetime64[ns]
 1   tpep_dropoff_datetime  datetime64[ns]
 2   passenger_count        float64       
 3   trip_distance          float64       
 4   store_and_fwd_flag     bool          
 5   payment_type           category      
 6   fare_amount            float64       
 7   extra                  float64       
 8   mta_tax                float64       
 9   tip_amount             float64       
 10  tolls_amount           float64       
 11  improvement_surcharge  float64       
 12  total_amount           float64       
 13  congestion_surcharge   float64       
 14  Airport_fee            float64       
 15  month                  int64         
 16  pickup_hr              int32         
 17  pickup_day             int32         
 18  pickup_month          